# 04 · Crop economics and the SQLite star schema
Builds the derived facts (`src/build_economics.py`), the data-quality tables, and loads everything into `database/indian_agri_revenue.db` (DDL in `sql/schema.sql`).

```
dim_crop ─┐                 ┌─ dim_state ── dim_location (district)
dim_year ─┼─ fact_district_crop_year      fact_state_crop_year ── fact_cost_of_cultivation
          ├─ fact_msp                      fact_weather_state
          └─ fact_market_price ── dim_market, dim_date
```

| | |
|---|---|
| **Objective** | Load everything into a SQLite star schema. |
| **Dataset** | Processed CSVs from notebooks 02-03. |
| **Business question** | Can the data be queried and modelled reliably? |
| **Method** | DDL in sql/schema.sql, foreign keys, row counts, integrity checks. |
| **Output** | database/indian_agri_revenue.db (6 dimensions, 8 facts). |
| **Key finding** | 0 foreign-key violations; every fact joins to its dimensions. |

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import RAW, CURATED, PROC, RESULTS, FIGURES, DB_PATH, connect_ro
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
BLUE, ORANGE, AQUA, YELLOW, PINK, RED, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#e34948", "#9aa19c"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "axes.titleweight": "bold", "axes.titlesize": 11})
def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", bbox_inches="tight"); plt.show()
from src import build_economics, build_quality, load_db

In [2]:
build_economics.main(); build_quality.main(); load_db.main()

{'district_crop_year': 199125, 'state_crop_year': 11479, 'msp_valued_share_district_rows': np.float64(0.614), 'profit_rows': 298, 'mh_mandi_valued_rows': 27, 'anomaly_rows': 5226}


            crop_name  dataset_value  official_value  diff_pct                   status
0                Rice         132.11          117.94      12.0         divergent (>10%)
1               Wheat         122.61          107.18      14.4         divergent (>10%)
2               Maize          20.47           28.98     -29.3         divergent (>10%)
3                Gram          10.13           10.90      -7.1  consistent (within 10%)
4            Soyabean          10.19           12.24     -16.7         divergent (>10%)
5  Rapeseed & Mustard           8.54            8.70      -1.9  consistent (within 10%)
6           Sugarcane         367.21          358.14       2.5  consistent (within 10%)
7              Cotton          33.17           36.05      -8.0  consistent (within 10%)
dim_crop                           26
dim_state                          36
dim_location                      760
dim_year                           37
dim_market                      3,418
dim_date          

fact_production               239,760


fact_district_crop_year       199,125


fact_state_crop_year           11,479
fact_msp                          516


fact_market_price             106,885
fact_cost_of_cultivation          300
fact_weather_subdivision        4,116
fact_weather_state                910
dq_issues                          18
dq_coverage                        67
dq_validation                       8
ref_sources                         9
foreign key violations: 0


database written to /home/claude/ifri/database/indian_agri_revenue.db


## 1. Tables and row counts

In [3]:
con = connect_ro()
tables = pd.read_sql("select name from sqlite_master where type='table' order by name", con).name
pd.DataFrame({"table": tables, "rows": [con.execute(f"select count(*) from {t}").fetchone()[0] for t in tables]})

,table,rows
0,dim_crop,26
1,dim_date,67
2,dim_location,760
3,dim_market,3418
4,dim_state,36
5,dim_year,37
6,dq_coverage,67
7,dq_issues,18
8,dq_validation,8
9,fact_cost_of_cultivation,300


In [4]:
print("foreign key violations:", len(con.execute("PRAGMA foreign_key_check").fetchall()))
pd.read_sql("PRAGMA table_info(fact_district_crop_year)", con)[["name", "type"]].T

foreign key violations: 0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
name,district_crop_year_id,location_id,state_id,crop_id,crop_name,crop_year,area_ha,production_apy_unit,production_price_qtl,n_seasons,yield_kg_per_ha,msp_effective,gross_value_msp_inr,gross_value_msp_per_ha,gross_value_at_2018_msp_inr,cpi_deflator_2018_100,gross_value_real_2018_inr,valuation_basis,is_provisional_year
type,INTEGER,INTEGER,INTEGER,INTEGER,TEXT,INTEGER,REAL,REAL,REAL,INTEGER,REAL,REAL,REAL,REAL,REAL,REAL,REAL,TEXT,INTEGER


## 2. How gross value is computed
`gross_value_msp_inr = production_price_qtl × msp_effective`. This is an estimate at the support price, **not farmer income**.

In [5]:
pd.read_sql("""
select l.state_name, l.district_name, c.crop_name, f.crop_year, round(f.area_ha) area_ha,
       round(f.production_price_qtl) qty_qtl, f.msp_effective, round(f.gross_value_msp_inr/1e7,1) gross_value_cr,
       round(f.gross_value_msp_per_ha) value_per_ha, f.valuation_basis
from fact_district_crop_year f join dim_location l using(location_id) join dim_crop c using(crop_id)
where l.district_name = 'Ludhiana' and f.crop_year = 2018 order by gross_value_cr desc""", con)

,state_name,district_name,crop_name,crop_year,area_ha,qty_qtl,msp_effective,gross_value_cr,value_per_ha,valuation_basis
0,Punjab,Ludhiana,Rice,2018,259000.0,17208955.0,1750.0,3011.6,116277.0,MSP-valued
1,Punjab,Ludhiana,Wheat,2018,251000.0,12880000.0,1840.0,2369.9,94419.0,MSP-valued
2,Punjab,Ludhiana,Sugarcane,2018,3000.0,2680000.0,275.0,73.7,245667.0,MSP-valued
3,Punjab,Ludhiana,Moong,2018,2600.0,22000.0,6975.0,15.3,59019.0,MSP-valued
4,Punjab,Ludhiana,Rapeseed & Mustard,2018,1300.0,25000.0,4200.0,10.5,80769.0,MSP-valued
5,Punjab,Ludhiana,Sunflower,2018,1000.0,19000.0,5388.0,10.2,102372.0,MSP-valued
6,Punjab,Ludhiana,Maize,2018,1300.0,51000.0,1700.0,8.7,66692.0,MSP-valued
7,Punjab,Ludhiana,Barley,2018,1200.0,42000.0,1440.0,6.0,50400.0,MSP-valued
8,Punjab,Ludhiana,Arhar/Tur,2018,600.0,6000.0,5675.0,3.4,56750.0,MSP-valued


## 3. Views for analysis

In [6]:
pd.read_sql("select * from vw_state_economics where crop_name='Wheat' and crop_year=2012 and c2_per_ha is not null", con).round(0)

,state_name,region,crop_name,crop_year,area_ha,yield_kg_per_ha,gross_value_msp_crore,gross_value_msp_per_ha,c2_per_ha,est_profit_c2_per_ha,est_profit_a2fl_per_ha,est_margin_c2_pct
0,Haryana,North,Wheat,2012,2496905.0,4452.0,15008.0,60106.0,54067.0,6039.0,28210.0,10.0
1,Madhya Pradesh,Central,Wheat,2012,5601025.0,2948.0,22289.0,39795.0,37120.0,2675.0,18822.0,7.0
2,Punjab,North,Wheat,2012,3517000.0,4724.0,22429.0,63773.0,49519.0,14254.0,37213.0,22.0
3,Rajasthan,West,Wheat,2012,3063202.0,3515.0,14535.0,47450.0,46942.0,508.0,17104.0,1.0
4,Uttar Pradesh,North,Wheat,2012,9733617.0,3219.0,42299.0,43456.0,44383.0,-927.0,15394.0,-2.0


In [7]:
pd.read_sql("select valuation_basis, count(*) rows from fact_district_crop_year group by 1", con)

,valuation_basis,rows
0,MSP-valued,122219
1,Not valued: no MSP (non-mandated crop),29812
2,Not valued: outside MSP window (2004-2019),47094


## Conclusion
The star schema is complete and consistent.

## Limitations
SQLite is single-user; fine for analysis, not for concurrent writes.

## Next steps
Move to Postgres if the data needs to be shared or refreshed on a schedule.